# Three-dimensional weak-symmetry mixed elasticity

The classical [Arnold–Falk–Winther family](https://arxiv.org/pdf/math/0701506) uses row-wise BDM stress, DG displacement and DG axial rotation. This original MHM integration retains all six local rigid modes with k≥2. It is not a reproduction of the two-dimensional enriched-family experiments. The solenoidal exact solution has the same body force for every Lamé λ.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/c9fe36ad0d1e253b3dfe5e7141938e9ab94b1121362e3e42339e9d977fb6cd54/68_mixed_elasticity3d-companion.zip"
COMPANION_SHA256 = "c9fe36ad0d1e253b3dfe5e7141938e9ab94b1121362e3e42339e9d977fb6cd54"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("elasticity/68_mixed_elasticity3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, Markdown, display



In [ ]:
folder = ROOT / "examples/results/mixed-elasticity3d"
record = json.loads((folder / "comparison.json").read_text())
assert len(record["convergence"]) == 10 and len(record["locking"]) == 7
assert record["source_independent_of_lambda"] and not record["source_changed_during_run"]
closed = dict(displacement_l2=3*np.pi/8, stress_l2=np.pi**2*np.sqrt(15/8), rotation_l2=np.pi**2*np.sqrt(15)/8)
assert max(abs(record["exact_norms"][k] - value) for k, value in closed.items()) < 8e-14
for row in record["convergence"]:
    path = folder / row["archive"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["archive_sha256"]
    with np.load(path) as field:
        assert hashlib.sha256(np.ascontiguousarray(field["basis"]).tobytes()).hexdigest() == row["basis_sha256"]
    assert max(row[key] for key in ("force_moment_max", "weak_symmetry_moment_max", "normal_traction_moment_max")) < 1e-9
header = "| Stress degree | n | Displacement % | Stress % | Rotation % |\n|---:|---:|---:|---:|---:|\n"
table = "\n".join(f"| {r['stress_degree']} | {r['n']} | {100*r['relative_errors']['displacement_l2']:.5g} | {100*r['relative_errors']['stress_l2']:.5g} | {100*r['relative_errors']['rotation_l2']:.5g} |" for r in record["convergence"])
display(Markdown(header + table))


Physical errors use separate exact-field denominators; stress is not symmetrized before comparison.

In [ ]:
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity3d/convergence.png")))

Fixed-space bulk-modulus control. Categorical λ positions avoid replacing infinity by a large finite value.

In [ ]:
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity3d/incompressibility.png")))

BDM2/P1/P1 fields evaluated with the archived basis matrix and physical orientations.

In [ ]:
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity3d/bdm2-fields.png")))

BDM3/P2/P2 fields use independent one-sided fine-cell polynomials, including nonsymmetric numerical stress.

In [ ]:
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity3d/bdm3-fields.png")))